In [ ]:
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score


# =========================================================
# 1. 데이터 불러오기
# =========================================================

train = pd.read_csv(
    "/kaggle/input/competitions/titanic/train.csv"
)

test = pd.read_csv(
    "/kaggle/input/competitions/titanic/test.csv"
)

y = train["Survived"].astype(int).to_numpy()

all_passengers = pd.concat(
    [
        train.drop(columns=["Survived"]),
        test
    ],
    ignore_index=True
)

number_of_train = len(train)
number_of_all = len(all_passengers)


# =========================================================
# 2. 기본 변수
# =========================================================

numeric_features = [
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked"
]

features = (
    numeric_features
    + categorical_features
)


# =========================================================
# 3. Tree 모델 전처리기
# =========================================================

tree_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        ),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]),
        categorical_features
    )
])


# =========================================================
# 4. HistGradientBoosting 전처리기
# =========================================================

dense_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        ),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]),
        categorical_features
    )
])


# =========================================================
# 5. LogisticRegression 전처리기
# =========================================================

logistic_preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]),
        categorical_features
    )
])


# =========================================================
# 6. 네 가지 모델 생성 함수
# =========================================================

def make_models():

    random_forest = Pipeline([
        (
            "preprocessor",
            tree_preprocessor
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=6,
                min_samples_split=5,
                min_samples_leaf=2,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    extra_trees = Pipeline([
        (
            "preprocessor",
            tree_preprocessor
        ),
        (
            "model",
            ExtraTreesClassifier(
                n_estimators=800,
                max_depth=8,
                min_samples_split=5,
                min_samples_leaf=2,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    hist_gradient_boosting = Pipeline([
        (
            "preprocessor",
            dense_preprocessor
        ),
        (
            "model",
            HistGradientBoostingClassifier(
                max_iter=250,
                learning_rate=0.05,
                max_leaf_nodes=15,
                min_samples_leaf=20,
                l2_regularization=1.0,
                random_state=42
            )
        )
    ])

    logistic_regression = Pipeline([
        (
            "preprocessor",
            logistic_preprocessor
        ),
        (
            "model",
            LogisticRegression(
                C=1.0,
                max_iter=2000,
                solver="liblinear",
                random_state=42
            )
        )
    ])

    return [
        random_forest,
        extra_trees,
        hist_gradient_boosting,
        logistic_regression
    ]


# =========================================================
# 7. 가족 키 생성
# =========================================================

def make_family_key(data):

    surname = (
        data["Name"]
        .str.split(",")
        .str[0]
        .str.strip()
    )

    family_size = (
        data["SibSp"]
        + data["Parch"]
        + 1
    )

    family_key = (
        surname
        + "_"
        + family_size.astype(str)
    )

    return family_key.where(
        family_size > 1,
        np.nan
    )


# =========================================================
# 8. 비대칭 가족 확률 혼합
# Master 강제 규칙 없음
# =========================================================

def mix_family_probability(
    source,
    source_y,
    target,
    model_probability
):

    source_family_key = make_family_key(
        source
    )

    target_family_key = make_family_key(
        target
    )

    source_protected_group = (
        (source["Sex"] == "female")
        | (source["Age"] < 16)
    )

    family_data = pd.DataFrame({
        "FamilyKey":
            source_family_key.to_numpy(),

        "Survived":
            source_y,

        "ProtectedGroup":
            source_protected_group.to_numpy()
    })

    family_statistics = (
        family_data[
            family_data["ProtectedGroup"]
        ]
        .dropna(
            subset=["FamilyKey"]
        )
        .groupby("FamilyKey")["Survived"]
        .agg(["sum", "count"])
    )

    global_survival_rate = np.mean(
        source_y
    )

    family_survival_rate = (
        family_statistics["sum"]
        + 2 * global_survival_rate
    ) / (
        family_statistics["count"]
        + 2
    )

    target_family_probability = (
        target_family_key.map(
            family_survival_rate
        )
    )

    target_protected_group = (
        (target["Sex"] == "female")
        | (target["Age"] < 16)
    )

    use_family_information = (
        target_protected_group
        & target_family_probability.notna()
    )

    final_probability = (
        model_probability.copy()
    )

    indices = np.where(
        use_family_information.to_numpy()
    )[0]

    final_probability[
        indices
    ] = (
        0.5
        * model_probability[
            indices
        ]
        + 0.5
        * target_family_probability.iloc[
            indices
        ].to_numpy()
    )

    return final_probability


# =========================================================
# 9. 네 모델의 확률을 현재 모델 방식으로 결합
# =========================================================

def combine_model_probabilities(
    probability_matrix
):

    binary_predictions = (
        probability_matrix >= 0.5
    ).astype(int)

    survival_votes = (
        binary_predictions.sum(axis=1)
    )

    # 3표 이상이면 생존
    hard_prediction = (
        survival_votes >= 3
    ).astype(int)

    # 2대2 동률이면 ExtraTrees 선택
    tie = (
        survival_votes == 2
    )

    hard_prediction[
        tie
    ] = binary_predictions[
        tie,
        1
    ]

    mean_probability = (
        probability_matrix.mean(axis=1)
    )

    # 하드 투표 결과와 확률 방향을 일치시킨다.
    prior_probability = np.where(
        hard_prediction == 1,
        np.maximum(
            mean_probability,
            0.51
        ),
        np.minimum(
            mean_probability,
            0.49
        )
    )

    return (
        prior_probability,
        hard_prediction
    )


# =========================================================
# 10. 기존 모델 OOF 확률 생성
# =========================================================

cross_validation = StratifiedKFold(
    n_splits=7,
    shuffle=True,
    random_state=42
)

oof_probability_matrix = np.zeros(
    (
        number_of_train,
        4
    )
)

fold_ids = np.zeros(
    number_of_train,
    dtype=int
)

for fold, (
    train_indices,
    validation_indices
) in enumerate(
    cross_validation.split(
        train,
        y
    )
):

    for model_index, model in enumerate(
        make_models()
    ):

        model.fit(
            train.iloc[
                train_indices
            ][features],

            y[
                train_indices
            ]
        )

        validation_probability = (
            model.predict_proba(
                train.iloc[
                    validation_indices
                ][features]
            )[:, 1]
        )

        validation_probability = (
            mix_family_probability(
                source=train.iloc[
                    train_indices
                ],

                source_y=y[
                    train_indices
                ],

                target=train.iloc[
                    validation_indices
                ],

                model_probability=(
                    validation_probability
                )
            )
        )

        oof_probability_matrix[
            validation_indices,
            model_index
        ] = validation_probability

    fold_ids[
        validation_indices
    ] = fold


oof_prior_probability, oof_prediction = (
    combine_model_probabilities(
        oof_probability_matrix
    )
)

print(
    "기존 모델 OOF 정확도:",
    accuracy_score(
        y,
        oof_prediction
    )
)


# =========================================================
# 11. test 기존 모델 확률 생성
# =========================================================

test_probability_matrix = np.zeros(
    (
        len(test),
        4
    )
)

for model_index, model in enumerate(
    make_models()
):

    model.fit(
        train[features],
        y
    )

    test_model_probability = (
        model.predict_proba(
            test[features]
        )[:, 1]
    )

    test_model_probability = (
        mix_family_probability(
            source=train,
            source_y=y,
            target=test,
            model_probability=(
                test_model_probability
            )
        )
    )

    test_probability_matrix[
        :,
        model_index
    ] = test_model_probability


test_prior_probability, baseline_prediction = (
    combine_model_probabilities(
        test_probability_matrix
    )
)

print(
    "기존 모델 test 생존 예측:",
    int(
        baseline_prediction.sum()
    )
)


# =========================================================
# 12. 그래프 그룹 키 생성
# =========================================================

surname = (
    all_passengers["Name"]
    .str.split(",")
    .str[0]
    .str.strip()
    .str.upper()
)

family_size = (
    all_passengers["SibSp"]
    + all_passengers["Parch"]
    + 1
)

graph_family_key = (
    surname
    + "_"
    + family_size.astype(str)
).where(
    family_size > 1,
    np.nan
)

graph_ticket_key = (
    all_passengers["Ticket"]
    .astype(str)
    .str.upper()
    .str.replace(
        r"\s+",
        "",
        regex=True
    )
)

graph_cabin_key = (
    all_passengers["Cabin"]
    .where(
        all_passengers["Cabin"].notna(),
        np.nan
    )
)

ticket_root = (
    graph_ticket_key
    .str.replace(
        r"\d$",
        "",
        regex=True
    )
)

graph_spte_key = (
    surname
    + "_"
    + all_passengers[
        "Pclass"
    ].astype(str)
    + "_"
    + ticket_root
    + "_"
    + all_passengers[
        "Embarked"
    ].fillna("U")
)

graph_keys = {
    "family":
        graph_family_key,

    "ticket":
        graph_ticket_key,

    "cabin":
        graph_cabin_key,

    "spte":
        graph_spte_key
}


# =========================================================
# 13. 여성·소년 보호집단
# =========================================================

title = (
    all_passengers["Name"]
    .str.extract(
        r",\s*([^.]*)\.",
        expand=False
    )
    .str.strip()
)

protected_group = (
    (all_passengers["Sex"] == "female")
    | (title == "Master")
).to_numpy()


# =========================================================
# 14. 그래프 인접행렬 생성
# =========================================================

def make_adjacency_matrix(
    selected_groups
):

    graph_matrix = np.zeros(
        (
            number_of_all,
            number_of_all
        ),
        dtype=float
    )

    group_weights = {
        "family": 1.0,
        "ticket": 1.0,
        "cabin": 0.35,
        "spte": 0.5
    }

    for group_name in selected_groups:

        group_series = graph_keys[
            group_name
        ]

        valid_groups = (
            group_series.dropna()
        )

        group_indices = (
            valid_groups.groupby(
                valid_groups
            ).groups
        )

        for indices in (
            group_indices.values()
        ):

            indices = np.asarray(
                list(indices),
                dtype=int
            )

            if len(indices) < 2:
                continue

            graph_matrix[
                np.ix_(
                    indices,
                    indices
                )
            ] += group_weights[
                group_name
            ]

    np.fill_diagonal(
        graph_matrix,
        0
    )

    # 여성·소년끼리의 연결은 강하게,
    # 그 외 연결은 약하게 반영한다.
    protected_pair = np.outer(
        protected_group,
        protected_group
    )

    graph_matrix *= np.where(
        protected_pair,
        1.0,
        0.15
    )

    row_sum = graph_matrix.sum(
        axis=1
    )

    normalized_matrix = np.divide(
        graph_matrix,
        row_sum[:, None],
        out=np.zeros_like(
            graph_matrix
        ),
        where=row_sum[:, None] > 0
    )

    return csr_matrix(
        normalized_matrix
    )


# =========================================================
# 15. 그래프 확률 전파 함수
# =========================================================

def propagate_probability(
    adjacency_matrix,
    known_mask,
    known_labels,
    prior_probability,
    alpha
):

    probability = (
        prior_probability.copy()
    )

    probability[
        known_mask
    ] = known_labels[
        known_mask
    ]

    has_neighbor = (
        np.asarray(
            adjacency_matrix.sum(
                axis=1
            )
        ).ravel() > 0
    )

    for _ in range(80):

        neighbor_probability = (
            np.asarray(
                adjacency_matrix
                @ probability
            )
            .ravel()
        )

        new_probability = (
            probability.copy()
        )

        unknown_mask = (
            ~known_mask
        )

        update_mask = (
            unknown_mask
            & has_neighbor
        )

        new_probability[
            update_mask
        ] = (
            (1 - alpha)
            * prior_probability[
                update_mask
            ]
            + alpha
            * neighbor_probability[
                update_mask
            ]
        )

        new_probability[
            known_mask
        ] = known_labels[
            known_mask
        ]

        if np.max(
            np.abs(
                new_probability
                - probability
            )
        ) < 1e-9:
            probability = (
                new_probability
            )
            break

        probability = (
            new_probability
        )

    return probability


# =========================================================
# 16. 그래프 구성 및 alpha 교차검증
# =========================================================

graph_configurations = {
    "family_ticket": [
        "family",
        "ticket"
    ],

    "plus_cabin": [
        "family",
        "ticket",
        "cabin"
    ],

    "plus_spte": [
        "family",
        "ticket",
        "spte"
    ],

    "all": [
        "family",
        "ticket",
        "cabin",
        "spte"
    ]
}

alpha_candidates = [
    0.0,
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7
]

all_prior_probability = np.concatenate([
    oof_prior_probability,
    test_prior_probability
])

calibrated_results = []


for configuration_name, selected_groups in (
    graph_configurations.items()
):

    adjacency_matrix = (
        make_adjacency_matrix(
            selected_groups
        )
    )

    for alpha in alpha_candidates:

        calibrated_oof_prediction = (
            np.zeros(
                number_of_train,
                dtype=int
            )
        )

        for fold in range(7):

            validation_mask = (
                fold_ids == fold
            )

            known_mask = np.zeros(
                number_of_all,
                dtype=bool
            )

            # 현재 validation Fold를 제외한
            # train 정답만 그래프에 제공한다.
            known_mask[
                :number_of_train
            ] = ~validation_mask

            known_labels = np.zeros(
                number_of_all,
                dtype=float
            )

            known_labels[
                :number_of_train
            ] = y

            propagated_probability = (
                propagate_probability(
                    adjacency_matrix=(
                        adjacency_matrix
                    ),

                    known_mask=known_mask,

                    known_labels=(
                        known_labels
                    ),

                    prior_probability=(
                        all_prior_probability
                    ),

                    alpha=alpha
                )
            )

            validation_scores = (
                propagated_probability[
                    :number_of_train
                ][
                    validation_mask
                ]
            )

            # 기존 모델이 해당 Fold에서 예측한
            # 생존자 수를 그대로 보존한다.
            number_of_survivors = int(
                oof_prediction[
                    validation_mask
                ].sum()
            )

            local_prediction = np.zeros(
                validation_mask.sum(),
                dtype=int
            )

            if number_of_survivors > 0:

                selected_indices = np.argsort(
                    validation_scores
                )[
                    -number_of_survivors:
                ]

                local_prediction[
                    selected_indices
                ] = 1

            calibrated_oof_prediction[
                validation_mask
            ] = local_prediction

        score = accuracy_score(
            y,
            calibrated_oof_prediction
        )

        calibrated_results.append({
            "configuration":
                configuration_name,

            "alpha":
                alpha,

            "accuracy":
                score
        })


# =========================================================
# 17. 가장 좋은 그래프 설정 선택
# =========================================================

calibrated_results = pd.DataFrame(
    calibrated_results
)

best_result = (
    calibrated_results
    .sort_values(
        [
            "accuracy",
            "alpha"
        ],
        ascending=[
            False,
            True
        ]
    )
    .iloc[0]
)

best_configuration = (
    best_result[
        "configuration"
    ]
)

best_alpha = float(
    best_result[
        "alpha"
    ]
)

best_cv_accuracy = float(
    best_result[
        "accuracy"
    ]
)

print(
    "선택된 그래프:",
    best_configuration
)

print(
    "선택된 alpha:",
    best_alpha
)

print(
    "그래프 OOF 정확도:",
    best_cv_accuracy
)


# =========================================================
# 18. 전체 train 정답을 이용한 test 그래프 전파
# =========================================================

final_adjacency_matrix = (
    make_adjacency_matrix(
        graph_configurations[
            best_configuration
        ]
    )
)

known_mask = np.zeros(
    number_of_all,
    dtype=bool
)

known_mask[
    :number_of_train
] = True

known_labels = np.zeros(
    number_of_all,
    dtype=float
)

known_labels[
    :number_of_train
] = y

final_prior_probability = np.concatenate([
    np.full(
        number_of_train,
        0.5
    ),
    test_prior_probability
])

final_graph_probability = (
    propagate_probability(
        adjacency_matrix=(
            final_adjacency_matrix
        ),

        known_mask=known_mask,

        known_labels=known_labels,

        prior_probability=(
            final_prior_probability
        ),

        alpha=best_alpha
    )
)

test_graph_scores = (
    final_graph_probability[
        number_of_train:
    ]
)


# =========================================================
# 19. 생존자 수 보존 Calibration
# =========================================================

number_of_test_survivors = int(
    baseline_prediction.sum()
)

final_prediction = np.zeros(
    len(test),
    dtype=int
)

selected_survivors = np.argsort(
    test_graph_scores
)[
    -number_of_test_survivors:
]

final_prediction[
    selected_survivors
] = 1


# =========================================================
# 20. 제출 파일 생성
# =========================================================

submission = pd.DataFrame({
    "PassengerId":
        test[
            "PassengerId"
        ].astype(int),

    "Survived":
        final_prediction.astype(int)
})

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)


# =========================================================
# 21. 결과 확인
# =========================================================

print()
print("=" * 55)
print("최종 그래프 혼합 결과")
print("=" * 55)

print(
    "기존 생존 예측:",
    int(
        baseline_prediction.sum()
    )
)

print(
    "최종 생존 예측:",
    int(
        final_prediction.sum()
    )
)

print(
    "기존 예측에서 변경된 승객:",
    int(
        (
            baseline_prediction
            != final_prediction
        ).sum()
    )
)

print(
    "저장 위치:",
    "/kaggle/working/submission.csv"
)

display(
    submission.head(10)
)